# Pairs Research Workflow

This notebook walks through the research flow for the current spread pipeline: data fetch, spread construction, stationarity checks, ARMA/GARCH fitting, and signal visualization.

In [ ]:
from datetime import datetime

import matplotlib.pyplot as plt

from pairs_trading.data.fetcher import fetch_pair_data
from pairs_trading.data.schemas import Asset, Pair
from pairs_trading.data.spread import (
    compute_spread,
    fit_arma_garch,
    check_cointegration,
)

In [ ]:
asset_a = Asset(symbol="MSFT", name="Microsoft")
asset_b = Asset(symbol="AAPL", name="Apple")
pair = Pair(asset_a=asset_a, asset_b=asset_b)

start = datetime(2022, 1, 1)
end = datetime(2023, 1, 1)

data_a, data_b = fetch_pair_data(asset_a, asset_b, start, end)
spread_data = compute_spread(pair, data_a, data_b, zscore_lookback=20)
spread_data.hedge_ratio, spread_data.half_life

In [ ]:
cointegration_result = check_cointegration(spread_data.spread)
cointegration_result

In [ ]:
arma_garch_result = fit_arma_garch(
    spread_data.spread,
    arma_candidates=[(1, 0, 0), (1, 0, 1)],
    garch_candidates=[(1, 1), (1, 2)],
)

arma_garch_result.arma_order, arma_garch_result.garch_order

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

spread_data.z_score.plot(ax=axes[0], title=f"Z-Score: {pair.pair_id}")
axes[0].axhline(0, color="black", linewidth=1)
axes[0].axhline(2, color="red", linestyle="--")
axes[0].axhline(-2, color="red", linestyle="--")

arma_garch_result.conditional_volatility.plot(
    ax=axes[1],
    color="darkorange",
    title="Conditional Volatility",
)

plt.tight_layout()